# Load Date Ideas From CSV

Этот ноутбук загружает идеи из `ideas.csv` в таблицу `date_ideas` в Postgres.

Что делает ноутбук:
- читает CSV из корня проекта
- нормализует строки
- подставляет `category` и `vibe`, если их нет в CSV
- вставляет записи в базу
- показывает итоговое количество идей


## 1. Setup

Перед запуском убедись, что:
- Postgres доступен
- миграции уже применены
- `.env` в корне проекта заполнен
- в окружении установлен `psycopg`


In [4]:
from __future__ import annotations

import csv
from pathlib import Path

import psycopg


In [5]:
def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / ".env").exists() and (candidate / "ideas.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find project root with .env and ideas.csv")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())

CSV_PATH = PROJECT_ROOT / "ideas.csv"
ENV_PATH = PROJECT_ROOT / ".env"

assert CSV_PATH.exists(), f"CSV not found: {CSV_PATH}"
assert ENV_PATH.exists(), f"Env file not found: {ENV_PATH}"

PROJECT_ROOT, CSV_PATH

(PosixPath('/Users/midas/Projects/this-is-it'),
 PosixPath('/Users/midas/Projects/this-is-it/ideas.csv'))

In [6]:
def load_env(path: Path) -> dict[str, str]:
    values: dict[str, str] = {}
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        values[key.strip()] = value.strip()
    return values


env = load_env(ENV_PATH)
dsn = (
    f"host={env['POSTGRES_HOST']} "
    f"port={env.get('POSTGRES_PORT', '5432')} "
    f"dbname={env['POSTGRES_DB']} "
    f"user={env['POSTGRES_USER']} "
    f"password={env['POSTGRES_PASSWORD']}"
)

dsn

'host=localhost port=5432 dbname=this-is-it user=postgres password=postgres'

## 2. Read and normalize CSV

Текущий CSV содержит только `title` и `description`, поэтому для загрузки в `date_ideas` мы добавляем значения по умолчанию:
- `category = "General"`
- `vibe = "Casual"`


In [7]:
DEFAULT_CATEGORY = "General"
DEFAULT_VIBE = "Casual"


def load_ideas(path: Path) -> list[dict[str, str]]:
    with path.open("r", encoding="utf-8", newline="") as file:
        reader = csv.DictReader(file)
        ideas: list[dict[str, str]] = []
        seen_titles: set[str] = set()

        for row in reader:
            title = (row.get("title") or "").strip()
            description = (row.get("description") or "").strip()
            category = (row.get("category") or DEFAULT_CATEGORY).strip() or DEFAULT_CATEGORY
            vibe = (row.get("vibe") or DEFAULT_VIBE).strip() or DEFAULT_VIBE

            if not title or not description:
                continue

            dedupe_key = title.casefold()
            if dedupe_key in seen_titles:
                continue

            seen_titles.add(dedupe_key)
            ideas.append(
                {
                    "title": title,
                    "description": description,
                    "category": category,
                    "vibe": vibe,
                }
            )

        return ideas


ideas = load_ideas(CSV_PATH)
len(ideas), ideas[:3]

(97,
 [{'title': 'Прогулка с кофе',
   'description': 'Купить кофе навынос и гулять по городу',
   'category': 'Город',
   'vibe': 'Легкое'},
  {'title': 'Пикник в парке',
   'description': 'Плед и легкие закуски на траве',
   'category': 'Природа',
   'vibe': 'Романтичное'},
  {'title': 'Фильм дома',
   'description': 'Выбрать фильм и устроить уютный вечер',
   'category': 'Дом',
   'vibe': 'Уютное'}])

## 3. Insert into Postgres

Ниже используется простая защита от дублей по `title`: перед вставкой читаются уже существующие заголовки, и новые строки добавляются только если такого `title` ещё нет.


In [8]:
INSERT_SQL = """
INSERT INTO date_ideas (title, description, category, vibe)
VALUES (%(title)s, %(description)s, %(category)s, %(vibe)s)
"""

SELECT_EXISTING_SQL = "SELECT title FROM date_ideas"
COUNT_SQL = "SELECT COUNT(*) FROM date_ideas"


with psycopg.connect(dsn) as conn:
    with conn.cursor() as cur:
        cur.execute(SELECT_EXISTING_SQL)
        existing_titles = {row[0].casefold() for row in cur.fetchall()}

        new_ideas = [idea for idea in ideas if idea['title'].casefold() not in existing_titles]

        if new_ideas:
            cur.executemany(INSERT_SQL, new_ideas)

        cur.execute(COUNT_SQL)
        total_count = cur.fetchone()[0]

    conn.commit()

print(f"Inserted: {len(new_ideas)}")
print(f"Total in table: {total_count}")

Inserted: 97
Total in table: 103


## 4. Verify loaded rows

Показываем последние добавленные идеи, чтобы быстро убедиться, что всё загрузилось корректно.


In [9]:
PREVIEW_SQL = """
SELECT id, title, category, vibe
FROM date_ideas
ORDER BY id DESC
LIMIT 10
"""

with psycopg.connect(dsn) as conn:
    with conn.cursor() as cur:
        cur.execute(PREVIEW_SQL)
        preview_rows = cur.fetchall()

preview_rows

[(103, 'Музыкальный вечер', 'Дом', 'Атмосферное'),
 (102, 'Кино-марафон', 'Дом', 'Уютное'),
 (101, 'Антикафе', 'Игры', 'Игривое'),
 (100, 'Книжный магазин', 'Культура', 'Уютное'),
 (99, 'Прогулка по мостам', 'Город', 'Атмосферное'),
 (98, 'Лодочная прогулка', 'Активность', 'Романтичное'),
 (97, 'Катамаран', 'Активность', 'Романтичное'),
 (96, 'Прокат самокатов', 'Активность', 'Энергичное'),
 (95, 'Закат и разговоры', 'Природа', 'Личное'),
 (94, 'День без плана', 'Город', 'Спонтанное')]